In [10]:
import plotly.graph_objects as go
import networkx as nx
import pandas as pd
from pathlib import Path

In [11]:
# 1) Read the respective graph: 
BASE_DIR = Path.cwd().parent
graph = nx.read_gexf(BASE_DIR / "notebooks" / "world_trade_network_petrol.gexf")

In [12]:
# 2) Get the dataframe with the coordinates of each country:
url = "https://gist.githubusercontent.com/tadast/8827699/raw/3cd639fa34eec5067080a61c69e3ae25e3076abb/countries_codes_and_coordinates.csv"
df_coords = pd.read_csv(url)

# Data cleaning:
df_coords = df_coords[['Alpha-3 code', 'Latitude (average)', 'Longitude (average)']].astype('string')
df_coords = df_coords.apply(lambda col: col.str.replace('"', '', regex=False).str.strip())
df_coords.rename({'Alpha-3 code': 'ISO3', 'Latitude (average)': 'LAT', 'Longitude (average)': 'LONG'}, axis=1, inplace=True)

# Data type adjustment:
df_coords.ISO3 = df_coords.ISO3.astype('category')
df_coords[['LAT', 'LONG']] = df_coords[['LAT', 'LONG']].astype('float32')

df_coords.head(3)

,ISO3,LAT,LONG
0,AFG,33.0,65.0
1,ALB,41.0,20.0
2,DZA,28.0,3.0


In [13]:
# 3) Filter the graph to avoid overloading the map (only routes over 500M USD) -> ADJUSTABLE:
graph = nx.DiGraph(((u, v, d) for u, v, d in graph.edges(data=True) if d['weight'] > 500000))

In [14]:
# 4) Create a coordinate dictionary for fast (O(1)) search:
coords_dict = df_coords.set_index('ISO3')[['LAT', 'LONG']].T.to_dict('list')

/tmp/ipykernel_22667/674997158.py:2: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  coords_dict = df_coords.set_index('ISO3')[['LAT', 'LONG']].T.to_dict('list')


In [15]:
# 5) Prepare the coordinate lists for the edges (trade flows):
edge_x = []
edge_y = []

for u, v, data in graph.edges(data=True):

    # Verify that both countries exist in the CSV of coordinates:
    if u in coords_dict and v in coords_dict:

        # Plotly uses (Longitude, Latitude) -> (X, Y):
        x0, y0 = coords_dict[u][1], coords_dict[u][0] 
        x1, y1 = coords_dict[v][1], coords_dict[v][0]
        
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

In [16]:
# 6) Prepare the lists for the nodes (countries):
node_x = []
node_y = []
node_text = []
node_sizes = []

for node in graph.nodes():

    if node in coords_dict:
        node_x.append(coords_dict[node][1])
        node_y.append(coords_dict[node][0])
        
        # Calculate the total exports from this country in the filtered network (out-weight):
        out_weight = sum([d['weight'] for _, _, d in graph.out_edges(node, data=True)])
        
        # Add the respective text: 
        node_text.append(f"{node}<br>Exports (M USD): {out_weight/1000:.1f}")
        
        # Scale the size of the node so that it visually fits on the map:
        node_sizes.append(max(out_weight / 800000, 3))

In [17]:
# 7) Build the interactive Plotly figure:
fig = go.Figure()

# Layer 1 - Trade routes lines:
fig.add_trace(go.Scattergeo(
    lon=edge_x, lat=edge_y,
    mode='lines',
    line=dict(width=0.8, color='rgba(30, 136, 229, 0.4)'), # Blue.
    hoverinfo='none'
))

# Layer 2 - Country nodes:
fig.add_trace(go.Scattergeo(
    lon=node_x, lat=node_y,
    mode='markers',
    hoverinfo='text',
    text=node_text,
    marker=dict(
        size=node_sizes,
        color='rgba(211, 47, 47, 0.8)', # Red.
        line=dict(width=1, color='black'),
        sizemode='area' # Circle area proportional to the value. 
    )
))

# 8) Configure map style and display it:
fig.update_layout(
    title_text='Geopolitics of Oil: Global Trade Routes (>500M USD)',
    showlegend=False,
    geo=dict(
        showland=True,
        landcolor="rgb(243, 243, 243)",
        countrycolor="rgb(204, 204, 204)",
        showocean=True,
        oceancolor="rgb(230, 240, 255)",
        projection_type="equirectangular" 
    ),
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()

In [18]:
# 9) Save it in html format: 
fig.write_html("world_trade_map.html", auto_open=True)